# Project 2 — Pneumonia Detection from Chest X-rays

**SciEncephalon AI · Summer Intern Series 2026**

## Goal
Train a deep-learning model that can look at a chest X-ray image and decide:
**Normal** vs **Pneumonia**. Build a small inference UI so a clinician (or curious teen!) can drag-drop an image and see what the model predicts.

> ⚠️ **Education Only.** Real clinical AI requires FDA clearance, prospective validation, and far more data than we have. Our goal is to *learn how* the pipeline works, not deploy it.

## Why this is a great project
- You'll learn **transfer learning** — the single most important trick in modern computer vision.
- You'll touch real medical imaging data and the messiness that comes with it.
- You'll learn to evaluate models with metrics that *actually matter in healthcare*: sensitivity (recall) and specificity, not just accuracy.

## Tech Stack
| Layer | Tool |
|---|---|
| Data | Kaggle "Chest X-Ray Images (Pneumonia)" by Paul Mooney |
| Modeling | PyTorch + a pretrained ResNet18 |
| Explainability | Grad-CAM (where did the model look?) |
| Visualization | **PyEcharts** — confusion matrix heatmap, training curves |
| UI (stretch) | Gradio |


## 1. Setup

In [1]:
# !pip install torch torchvision pyecharts numpy pillow scikit-learn --quiet
import os, math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image, ImageDraw
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

from pyecharts import options as opts
from pyecharts.charts import Line, HeatMap, Bar, Liquid, Page
from pyecharts.commons.utils import JsCode

torch.manual_seed(42); random.seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


## 2. Get the Data — Real or Synthetic Fallback

**Real path:** Download the Kaggle dataset and point `DATA_DIR` to it. Folder layout:
```
chest_xray/train/{NORMAL,PNEUMONIA}/*.jpeg
chest_xray/val/{NORMAL,PNEUMONIA}/*.jpeg
chest_xray/test/{NORMAL,PNEUMONIA}/*.jpeg
```

**Synthetic fallback (so this notebook always runs):** We generate fake "X-ray-like" greyscale images. They are useless clinically, but let us *prove the pipeline runs*. When you swap in real Kaggle data, only `DATA_DIR` changes.

In [2]:
DATA_DIR = Path("./synthetic_xray")
DATA_DIR.mkdir(exist_ok=True)

def make_fake_xray(label: str, idx: int):
    """Generate a 224x224 greyscale image. PNEUMONIA = blotchy white spots; NORMAL = clean."""
    img = Image.new("L", (224, 224), color=20)
    draw = ImageDraw.Draw(img)
    # Add ribcage-like vertical streaks
    for i in range(20, 220, 25):
        draw.line([(i, 30), (i, 200)], fill=110, width=2)
    if label == "PNEUMONIA":
        # blotchy white opacities
        rng = np.random.default_rng(idx)
        for _ in range(rng.integers(4, 9)):
            x, y = rng.integers(40, 184, 2)
            r = int(rng.integers(8, 22))
            draw.ellipse([x-r, y-r, x+r, y+r], fill=int(rng.integers(180, 240)))
    return img

def build_synth(split, n_per_class=80):
    for cls in ["NORMAL", "PNEUMONIA"]:
        d = DATA_DIR / split / cls; d.mkdir(parents=True, exist_ok=True)
        for i in range(n_per_class):
            make_fake_xray(cls, i + (0 if split == "train" else 10000)).save(d / f"{cls}_{i:03d}.png")

if not (DATA_DIR / "train").exists():
    for sp, n in [("train", 80), ("val", 20), ("test", 30)]:
        build_synth(sp, n)
print("Synthetic dataset ready at:", DATA_DIR.resolve())
print({sp: {c: len(list((DATA_DIR/sp/c).glob('*.png'))) for c in ['NORMAL','PNEUMONIA']}
       for sp in ['train','val','test']})

Synthetic dataset ready at: /Users/ndingari/Dropbox/SciEncephalon/Internship SJ/2026/intern_projects/synthetic_xray
{'train': {'NORMAL': 80, 'PNEUMONIA': 80}, 'val': {'NORMAL': 20, 'PNEUMONIA': 20}, 'test': {'NORMAL': 30, 'PNEUMONIA': 30}}


## 3. Dataset & DataLoaders

In [3]:
class XrayDS(Dataset):
    def __init__(self, root, train=False):
        self.items = []
        for cls_idx, cls in enumerate(["NORMAL", "PNEUMONIA"]):
            for p in (root / cls).glob("*.png"):
                self.items.append((p, cls_idx))
        if train:
            self.tf = transforms.Compose([
                transforms.Grayscale(num_output_channels=3),
                transforms.RandomHorizontalFlip(),
                transforms.RandomRotation(8),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])
        else:
            self.tf = transforms.Compose([
                transforms.Grayscale(num_output_channels=3),
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, y = self.items[i]
        return self.tf(Image.open(p).convert("L")), y

train_ds = XrayDS(DATA_DIR/"train", train=True)
val_ds   = XrayDS(DATA_DIR/"val")
test_ds  = XrayDS(DATA_DIR/"test")

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=16)
test_dl  = DataLoader(test_ds,  batch_size=16)
print("Sizes:", len(train_ds), len(val_ds), len(test_ds))

Sizes: 160 40 60


## 4. Build the Model — Transfer Learning

**The big idea**: ResNet18 was trained on millions of natural images (ImageNet). The early layers already learned how to detect edges, textures, and shapes. We freeze those, swap out the final layer for our 2-class problem, and train only the new head. This is **transfer learning** — fast, data-efficient, and surprisingly powerful.

In [4]:
def build_model():
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    for p in m.parameters(): p.requires_grad = False        # freeze backbone
    m.fc = nn.Linear(m.fc.in_features, 2)                   # new head
    return m.to(DEVICE)

model = build_model()
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

def evaluate(dl):
    model.eval()
    ys, preds, probs = [], [], []
    with torch.no_grad():
        for x, y in dl:
            x = x.to(DEVICE); logits = model(x)
            p = torch.softmax(logits, 1)[:, 1].cpu().numpy()
            ys += y.tolist(); preds += logits.argmax(1).cpu().tolist(); probs += p.tolist()
    return np.array(ys), np.array(preds), np.array(probs)

## 5. Train (a few epochs are enough on synthetic data)

In [5]:
EPOCHS = 5
history = {"train_loss": [], "val_acc": []}
for ep in range(EPOCHS):
    model.train(); running = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); logits = model(x); loss = loss_fn(logits, y)
        loss.backward(); opt.step(); running += loss.item() * x.size(0)
    train_loss = running / len(train_ds)
    yv, pv, _ = evaluate(val_dl)
    val_acc = (yv == pv).mean()
    history["train_loss"].append(round(train_loss, 4))
    history["val_acc"].append(round(float(val_acc), 4))
    print(f"Epoch {ep+1}: train_loss={train_loss:.4f}  val_acc={val_acc:.3f}")

Epoch 1: train_loss=0.4800  val_acc=1.000
Epoch 2: train_loss=0.2671  val_acc=1.000
Epoch 3: train_loss=0.1833  val_acc=1.000
Epoch 4: train_loss=0.1208  val_acc=1.000
Epoch 5: train_loss=0.0994  val_acc=1.000


## 6. Visualize Training with PyEcharts

In [6]:
xs = [f"E{i+1}" for i in range(EPOCHS)]

line = (
    Line(init_opts=opts.InitOpts(width="800px", height="420px"))
    .add_xaxis(xs)
    .add_yaxis("Train Loss", history["train_loss"], yaxis_index=0,
               linestyle_opts=opts.LineStyleOpts(width=3, color="#ef4444"))
    .add_yaxis("Val Accuracy", history["val_acc"], yaxis_index=1,
               linestyle_opts=opts.LineStyleOpts(width=3, color="#10b981"))
    .extend_axis(yaxis=opts.AxisOpts(name="Val Accuracy", min_=0, max_=1, position="right"))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Training Progress",
                                   subtitle="Loss should fall, accuracy should rise"),
        yaxis_opts=opts.AxisOpts(name="Train Loss"),
        legend_opts=opts.LegendOpts(pos_top="8%"),
    )
)
line.render_notebook()

## 7. Test Set Evaluation — The Metrics That Matter

In [7]:
yt, pt, probs = evaluate(test_dl)
cm = confusion_matrix(yt, pt)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) else 0   # recall on positives
specificity = tn / (tn + fp) if (tn + fp) else 0
auc = roc_auc_score(yt, probs) if len(set(yt)) > 1 else float("nan")

print(classification_report(yt, pt, target_names=["NORMAL","PNEUMONIA"], zero_division=0))
print(f"Sensitivity (catch sick): {sensitivity:.3f}")
print(f"Specificity (skip healthy): {specificity:.3f}")
print(f"AUC: {auc:.3f}")

              precision    recall  f1-score   support

      NORMAL       1.00      1.00      1.00        30
   PNEUMONIA       1.00      1.00      1.00        30

    accuracy                           1.00        60
   macro avg       1.00      1.00      1.00        60
weighted avg       1.00      1.00      1.00        60

Sensitivity (catch sick): 1.000
Specificity (skip healthy): 1.000
AUC: 1.000


In [8]:
# Confusion-matrix heatmap
labels = ["NORMAL", "PNEUMONIA"]
data = [[i, j, int(cm[i, j])] for i in range(2) for j in range(2)]

heat = (
    HeatMap(init_opts=opts.InitOpts(width="520px", height="420px"))
    .add_xaxis(labels)
    .add_yaxis("predicted", labels, data,
               label_opts=opts.LabelOpts(is_show=True, color="#fff"))
    .set_global_opts(
        title_opts=opts.TitleOpts(title="Confusion Matrix",
                                   subtitle="Rows = true label · Cols = predicted"),
        visualmap_opts=opts.VisualMapOpts(min_=0, max_=int(cm.max()),
                                           range_color=["#dbeafe","#1e40af"]),
    )
)
heat.render_notebook()

In [9]:
# Liquid fill chart for the headline metric — fun & informative
liquid = (
    Liquid(init_opts=opts.InitOpts(width="380px", height="380px"))
    .add("Sensitivity", [sensitivity], is_outline_show=False, shape="circle")
    .set_global_opts(title_opts=opts.TitleOpts(title=f"Sensitivity = {sensitivity:.0%}",
                                                 subtitle="Fraction of pneumonia cases caught"))
)
liquid.render_notebook()

## 8. Inference Demo (drop-in for Gradio)

Wrap the model in a single `predict_image(path)` function — exactly what a Gradio interface needs.

In [10]:
tf_eval = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

@torch.no_grad()
def predict_image(image_or_path):
    img = Image.open(image_or_path).convert("L") if isinstance(image_or_path, (str, Path)) else image_or_path
    x = tf_eval(img).unsqueeze(0).to(DEVICE)
    p = torch.softmax(model(x), 1)[0].cpu().numpy()
    return {"NORMAL": float(p[0]), "PNEUMONIA": float(p[1])}

sample = next((DATA_DIR/"test"/"PNEUMONIA").glob("*.png"))
print("Sample image:", sample.name)
print("Prediction:", predict_image(sample))

Sample image: PNEUMONIA_022.png
Prediction: {'NORMAL': 0.004253108985722065, 'PNEUMONIA': 0.995746910572052}


## 9. Wrap-up

### What you built
- A full **PyTorch transfer-learning pipeline**: dataset → model → train → eval.
- Healthcare-aware metrics: sensitivity, specificity, AUC.
- Interactive PyEcharts dashboard: training curves, confusion heatmap, liquid metric gauge.

### Stretch goals
1. **Real Kaggle data** — point `DATA_DIR` at the real dataset and re-run.
2. **Grad-CAM**: visualize *where* the model is looking. Library: `pytorch-grad-cam`.
3. **Class-imbalance fix**: the Kaggle dataset is ~75% pneumonia. Add `WeightedRandomSampler`.
4. **Multi-class**: add COVID class from the COVID-19 Radiography DB.
5. **Gradio UI**: 10 lines of code wraps `predict_image` into a shareable web app.

### Healthcare AI Ethics
- Synthetic data ≠ clinical data. Talk about this up front.
- Always report sensitivity/specificity, not just accuracy.
- Document **failure modes** (e.g., poor performance on pediatric vs adult films).
- Build a model card before sharing the demo widely.
